*  DSC530-T301 Data Exploration and Analysis (2263-1)
*  Weeks 5 & 6 Coding Assignment
*  David Jorgensen

# Hands-on data analysis with Pandas

# Chapter 4, Exercise 1

With the earthquakes.csv file, select all the earthquakes in Japan with a magnitude of 4.9 or greater using the mb magnitude type.

In [23]:
import pandas as pd
import numpy as np

# Load ./textbook/ch_04/exercises/earthquakes.csv into a data frame named 'eq_data'
eq_data = pd.read_csv("textbook/ch_04/exercises/earthquakes.csv")

# Select all the earthquakes in Japan with a magnitude of 4.9 or greater using the mb magnitude type
japan_eqs = eq_data[(eq_data['parsed_place'].str.lower() == 'japan') & (eq_data['magType'] == 'mb') & (eq_data['mag'] >= 4.9)]
print(japan_eqs)

      mag magType           time                         place  tsunami  \
1563  4.9      mb  1538977532250  293km ESE of Iwo Jima, Japan        0   
2576  5.4      mb  1538697528010    37km E of Tomakomai, Japan        0   
3072  4.9      mb  1538579732490     15km ENE of Hasaki, Japan        0   
3632  4.9      mb  1538450871260    53km ESE of Hitachi, Japan        0   

     parsed_place  
1563        Japan  
2576        Japan  
3072        Japan  
3632        Japan  


## Chapter 4, Exercise 2

Create bins for each full number of earthquake magnitude (for instance, the first bin is (0,1], the second is (1,2], and so on) with the ml magnitude type and count how many are in each bin.

In [24]:
# Filter for ml magnitude type and drop missing magnitudes
ml_eqs = eq_data[(eq_data['magType'] == 'ml') & (eq_data['mag'].notna())]

# Determine bin edges for magnitudes
start = int(min(0, np.floor(ml_eqs['mag'].min())))
end = int(np.ceil(ml_eqs['mag'].max()))
bins = np.arange(start, end + 1)  # edges: start, start+1, ..., end

if len(bins) < 2:
    print("Not enough range to create magnitude bins.")
else:
    # Bin magnitudes into intervals like (0,1], (1,2], ...
    binned = pd.cut(ml_eqs['mag'], bins=bins, right=True)
    # Count how many are in each bin (preserves bin order)
    counts = binned.value_counts(sort=False)
    print(counts)

mag
(-2, -1]      13
(-1, 0]      491
(0, 1]      2207
(1, 2]      3105
(2, 3]       862
(3, 4]       122
(4, 5]         2
(5, 6]         1
Name: count, dtype: int64


## Chapter 4, Exercise 4

Build a crosstab with the earthquake data between the tsunami column and the magType column. Rather than showing the frequency count, show the maximum magnitude that was observed for each combination. Put the magnitude type along the columns.

In [25]:
# Build a crosstab with the earthquake data between the tsunami column and the magType column
tsunami_crosstab = pd.crosstab(eq_data['tsunami'], eq_data['magType'])

# Show maximum magnitude observed for each combination and magnitude type
max_magnitudes = eq_data.pivot_table(index='tsunami', columns='magType', values='mag', aggfunc='max')
print(max_magnitudes)

magType   mb  mb_lg    md   mh   ml  ms_20    mw  mwb  mwr  mww
tsunami                                                        
0        5.6    3.5  4.11  1.1  4.2    NaN  3.83  5.8  4.8  6.0
1        6.1    NaN   NaN  NaN  5.1    5.7  4.41  NaN  NaN  7.5


## Chapter 4, Exercise 6

Create a pivot table of the FAANG data that compares the stocks. Put the ticker in the rows and show the averages of the OHLC and volume traded data.

In [26]:
# Import FAANG data
faang_data = pd.read_csv("textbook/ch_04/exercises/FAANG.csv")

# Create a pivot table with ticker as rows, and show averages of the OHLC and volume traded columns
faang_pivot = faang_data.pivot_table(index='ticker', values=['open', 'high', 'low', 'close', 'volume'], aggfunc='mean')
print(faang_pivot)

              close         high          low         open        volume
ticker                                                                  
AAPL      47.263357    47.748526    46.795877    47.277859  1.360803e+08
AMZN    1641.726176  1662.839839  1619.840519  1644.072709  5.648994e+06
FB       171.510956   173.613347   169.303148   171.472948  2.765860e+07
GOOG    1113.225134  1125.777606  1101.001658  1113.554101  1.741965e+06
NFLX     319.290319   325.219322   313.187330   319.620558  1.146962e+07


## Chapter 4, Exercise 7

Calculate the z-scores for each numeric column of Amazon’s data (ticker is AMZN) in Q4 2018 using apply().

In [27]:
# Calculate z-scores for each numeric column of AMZN in Q4 2018 using apply
# Define Q4 2018 date range
amzn_q4_2018 = faang_data[(faang_data['ticker'] == 'AMZN') & (faang_data['date'] >= '2018-10-01') & (faang_data['date'] <= '2018-12-31')]
# Calculate z-scores
z_scores = amzn_q4_2018.select_dtypes(include=[np.number]).apply(lambda x: (x - x.mean()) / x.std())
print(z_scores)

         high       low      open     close    volume
690  2.368006  2.502113  2.337813  2.385848 -1.630411
691  2.227302  2.247433  2.190795  2.155037 -0.861879
692  2.058955  2.139987  2.068570  2.025489 -0.920345
693  1.819474  1.781561  1.850048  1.722816 -0.126582
694  1.628173  1.554416  1.642819  1.584748 -0.298771
..        ...       ...       ...       ...       ...
748 -2.159820 -2.187566 -2.179582 -2.226185 -0.141238
749 -1.611714 -1.810493 -2.026617 -1.339674  1.123063
750 -1.641276 -1.626703 -1.456521 -1.404343  0.849827
751 -1.325261 -1.231588 -1.328549 -1.289951  0.496102
752 -1.273456 -0.975763 -1.078283 -1.122691 -0.246405

[63 rows x 5 columns]


## Chapter 4, Exercise 10

The [European Centre for Disease Prevention and Control (ECDC)](https://www.ecdc.europa.eu/en/publications-data/download-todays-data-geographic-distribution-covid-19-cases-worldwide) provides an open dataset on COVID-19 cases called daily number of new reported cases of COVID-19 by country worldwide. This dataset is updated daily, but we will use a snapshot that contains data through September 18, 2020. Complete the following tasks to practice the skills you've learned up to this point in the book:

1. Prepare the data
    - Read in the data in the covid19_cases.csv file.
    - Create a date column by parsing the dateRep column into a datetime.
    - Set the date column as the index.
    - Use the replace() method to update all occurrences of United_States_of_America and United_Kingdom to USA and UK, respectively.
    - Sort the index.
2. For the five countries with the most cases (cumulative), find the day with the largest number of cases.
3. Find the 7-day average change in COVID-19 cases for the last week in the data for the five countries with the most cases.
4. Find the first date that each country other than China had cases.
5. Rank the countries by cumulative cases using percentiles.

In [28]:
# Read in covid19_cases.csv
covid_data = pd.read_csv("textbook/ch_04/exercises/covid19_cases.csv")
# Create a date column by parsing the dateRep column into a datetime
covid_data['date'] = pd.to_datetime(covid_data['dateRep'], format='%d/%m/%Y')
# Set the date column as the index
covid_data.set_index('date', inplace=True)
# Use the replace method to update all occurrences of United_States_of_America and United_Kingdom to USA and UK, respectively
covid_data['countriesAndTerritories'] = covid_data['countriesAndTerritories'].replace('United_States_of_America', 'USA')
covid_data['countriesAndTerritories'] = covid_data['countriesAndTerritories'].replace('United_Kingdom', 'UK')
# Sort index in ascending order
covid_data.sort_index(inplace=True)
print(covid_data.head())

               dateRep  day  month  year  cases  deaths  \
date                                                      
2019-12-31  31/12/2019   31     12  2019      0       0   
2019-12-31  31/12/2019   31     12  2019      0       0   
2019-12-31  31/12/2019   31     12  2019      0       0   
2019-12-31  31/12/2019   31     12  2019      0       0   
2019-12-31  31/12/2019   31     12  2019      0       0   

           countriesAndTerritories geoId countryterritoryCode  popData2019  \
date                                                                         
2019-12-31                 Belgium    BE                  BEL   11455519.0   
2019-12-31                  Mexico    MX                  MEX  127575529.0   
2019-12-31                 Ecuador    EC                  ECU   17373657.0   
2019-12-31                  Russia    RU                  RUS  145872260.0   
2019-12-31             Netherlands    NL                  NLD   17282163.0   

           continentExp  \
date        

In [29]:
# For the five countries with the most cases (cumulative), find the day with the largest number of cases
# Identify top 5 countries by cumulative cases
top5_countries = covid_data.groupby('countriesAndTerritories')['cases'].sum().nlargest(5).index
for country in top5_countries:
    # Find the day with the largest number of cases
    country_data = covid_data[covid_data['countriesAndTerritories'] == country]
    max_cases_day = country_data.loc[country_data['cases'].idxmax()]
    print(f"{country}: {max_cases_day.name.date()} with {max_cases_day['cases']} cases")

USA: 2020-07-25 with 78427 cases
India: 2020-09-17 with 97894 cases
Brazil: 2020-07-30 with 69074 cases
Russia: 2020-07-18 with 12640 cases
Peru: 2020-08-17 with 10143 cases


In [30]:
# Find the 7-day average change in COVID-19 cases for the last week in the data for the five countries with the most cases
for country in top5_countries:
    country_data = covid_data[covid_data['countriesAndTerritories'] == country]
    country_data = country_data.sort_index()
    # Calculate daily change and then 7-day rolling average of that change
    country_data['7_day_avg_change'] = country_data['cases'].diff().rolling(window=7).mean()
    last_week_avg_change = country_data['7_day_avg_change'].iloc[-7:]
    print(f"{country} - Last 7 days average change in cases:\n{last_week_avg_change}\n")

USA - Last 7 days average change in cases:
date
2020-09-13    -474.285714
2020-09-14     473.714286
2020-09-15    1513.000000
2020-09-16    3478.714286
2020-09-17   -1047.000000
2020-09-18     865.714286
2020-09-19     306.857143
Name: 7_day_avg_change, dtype: float64

India - Last 7 days average change in cases:
date
2020-09-13     534.285714
2020-09-14     181.285714
2020-09-15    1142.857143
2020-09-16      59.571429
2020-09-17     308.428571
2020-09-18     -18.142857
2020-09-19    -604.714286
Name: 7_day_avg_change, dtype: float64

Brazil - Last 7 days average change in cases:
date
2020-09-13     479.285714
2020-09-14      35.285714
2020-09-15     697.428571
2020-09-16    3196.285714
2020-09-17     143.428571
2020-09-18    -607.714286
2020-09-19    -560.142857
Name: 7_day_avg_change, dtype: float64

Russia - Last 7 days average change in cases:
date
2020-09-13     40.428571
2020-09-14     36.285714
2020-09-15     46.285714
2020-09-16     61.428571
2020-09-17    810.000000
2020-09-1

In [31]:
# Find the first date that each country other than China had cases
# Get unique countries
countries = covid_data['countriesAndTerritories'].unique()
# Iterate through countries and find first case date
for country in countries:
    if country != 'China':
        country_data = covid_data[covid_data['countriesAndTerritories'] == country]
        first_case_date = country_data[country_data['cases'] > 0].index.min()
        print(f"{country}: {first_case_date.date() if pd.notna(first_case_date) else 'No cases recorded'}")

Belgium: 2020-02-04


Mexico: 2020-02-29
Ecuador: 2020-03-01
Russia: 2020-02-01
Netherlands: 2020-02-28
UK: 2020-02-01
Germany: 2020-01-28
Denmark: 2020-02-27
Japan: 2020-01-15
Sweden: 2020-02-01
Indonesia: 2020-03-02
Canada: 2020-01-26
South_Korea: 2020-01-20
Cases_on_an_international_conveyance_Japan: 2020-02-05
Afghanistan: 2020-02-25
Monaco: 2020-02-29
Italy: 2020-01-31
Spain: 2020-02-01
Croatia: 2020-02-26
Australia: 2020-01-25
New_Zealand: 2020-02-28
Finland: 2020-01-30
United_Arab_Emirates: 2020-01-27
Bahrain: 2020-02-24
Thailand: 2020-01-13
Norway: 2020-02-27
Belarus: 2020-02-28
Dominican_Republic: 2020-03-02
Oman: 2020-02-25
Malaysia: 2020-01-25
Austria: 2020-02-26
USA: 2020-01-21
Nigeria: 2020-02-28
Cambodia: 2020-01-28
Sri_Lanka: 2020-01-28
Lithuania: 2020-02-28
India: 2020-01-30
Kuwait: 2020-02-24
North_Macedonia: 2020-02-27
Greece: 2020-02-27
Nepal: 2020-01-25
Iraq: 2020-02-25
Israel: 2020-02-22
Ireland: 2020-03-01
Estonia: 2020-02-28
Philippines: 2020-01-30
Lebanon: 2020-02-22
Luxembourg: 2020

In [32]:
# Rank the countries by cumulative cases using percentiles
# Calculate cumulative cases per country
cumulative_cases = covid_data.groupby('countriesAndTerritories')['cases'].sum()
# Calculate percentile ranks
percentile_ranks = cumulative_cases.rank(pct=True)
print(percentile_ranks)

countriesAndTerritories
Afghanistan       0.690476
Albania           0.561905
Algeria           0.733333
Andorra           0.271429
Angola            0.404762
                    ...   
Vietnam           0.228571
Western_Sahara    0.219048
Yemen             0.300000
Zambia            0.576190
Zimbabwe          0.500000
Name: cases, Length: 210, dtype: float64
